# Order-to-Delivery Reconciliation Validation

This notebook validates the order-to-delivery reconciliation pipeline
after transformation in dbt.

The analysis focuses on:
- data grain and delivery coverage;
- reconciliation exception counts;
- overlap between exception categories;
- financial relevance of exceptions;
- final dashboard mart quality checks.

In [1]:
import duckdb

In [2]:
con = duckdb.connect("../reconciliation.duckdb")

## 1. Data sanity checks

First, validate the grain of the source-level staging models.

`stg_orders` should contain one row per order, while `stg_deliveries`
may contain multiple records for the same order because duplicate
delivery records are intentionally present in the synthetic dataset.

In [3]:
con.sql("""SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(amount) AS total_order_amount
FROM stg_orders
""").show()

┌──────────────┬───────────────┬────────────────────┐
│ total_orders │ unique_orders │ total_order_amount │
│    int64     │     int64     │   decimal(38,2)    │
├──────────────┼───────────────┼────────────────────┤
│         1000 │          1000 │          263692.16 │
└──────────────┴───────────────┴────────────────────┘



In [4]:
con.sql("""SELECT
    COUNT(*) AS delivery_records,
    COUNT(DISTINCT order_id) AS unique_order_ids
FROM stg_deliveries
""").show()

┌──────────────────┬──────────────────┐
│ delivery_records │ unique_order_ids │
│      int64       │      int64       │
├──────────────────┼──────────────────┤
│              992 │              956 │
└──────────────────┴──────────────────┘



The orders table preserves a one-row-per-order grain. The deliveries
table contains repeated order IDs, which is expected because duplicate
delivery records are part of the reconciliation test scenarios.

## 2. Reconciliation exceptions

The reconciliation pipeline classifies orders into three exception
categories: missing delivery, amount discrepancy, and duplicate delivery.

The following checks validate both category counts and whether the
exception groups overlap.

In [5]:
con.sql("""SELECT
    'Missing Delivery' AS exception_type,
    COUNT(DISTINCT order_id) AS affected_orders
FROM mart_missing_deliveries

UNION ALL

SELECT
    'Amount Discrepancy',
    COUNT(DISTINCT order_id)
FROM mart_amount_discrepancies

UNION ALL

SELECT
    'Duplicate Delivery',
    COUNT(DISTINCT order_id)
FROM mart_duplicate_deliveries
""").show()

┌────────────────────┬─────────────────┐
│   exception_type   │ affected_orders │
│      varchar       │      int64      │
├────────────────────┼─────────────────┤
│ Missing Delivery   │              44 │
│ Amount Discrepancy │              57 │
│ Duplicate Delivery │              36 │
└────────────────────┴─────────────────┘



In [6]:
con.sql("""SELECT
    COUNT(DISTINCT a.order_id) AS overlap_orders
FROM mart_amount_discrepancies a
INNER JOIN mart_duplicate_deliveries d
    ON a.order_id = d.order_id
""").show()

┌────────────────┐
│ overlap_orders │
│     int64      │
├────────────────┤
│              0 │
└────────────────┘



In [7]:
con.sql("""SELECT COUNT(DISTINCT order_id) AS unique_exception_orders
FROM (
    SELECT order_id FROM mart_missing_deliveries

    UNION

    SELECT order_id FROM mart_amount_discrepancies

    UNION

    SELECT order_id FROM mart_duplicate_deliveries
)
""").show()

┌─────────────────────────┐
│ unique_exception_orders │
│          int64          │
├─────────────────────────┤
│                     137 │
└─────────────────────────┘



The exception groups are mutually exclusive in the generated dataset.
The total number of unique exception orders therefore equals the sum
of the three exception categories.

## 3. Financial relevance

Exception counts alone do not show materiality. For amount discrepancies,
the absolute difference is used so positive and negative mismatches do
not cancel each other out.

In [8]:
con.sql("""
SELECT
    COUNT(*) AS discrepancy_count,
    ROUND(SUM(absolute_difference), 2) AS discrepancy_volume,
    ROUND(AVG(absolute_difference), 2) AS avg_discrepancy,
    ROUND(MAX(absolute_difference), 2) AS max_discrepancy
FROM mart_amount_discrepancies
""").show()

┌───────────────────┬────────────────────┬─────────────────┬─────────────────┐
│ discrepancy_count │ discrepancy_volume │ avg_discrepancy │ max_discrepancy │
│       int64       │   decimal(38,2)    │     double      │  decimal(18,2)  │
├───────────────────┼────────────────────┼─────────────────┼─────────────────┤
│                57 │            3106.54 │            54.5 │          162.67 │
└───────────────────┴────────────────────┴─────────────────┴─────────────────┘



In [9]:
con.sql("""SELECT
    COUNT(*) AS missing_orders,
    ROUND(SUM(order_amount), 2) AS affected_order_value,
    ROUND(AVG(order_amount), 2) AS avg_order_value,
    ROUND(MAX(order_amount), 2) AS largest_order_value
FROM mart_missing_deliveries
""").show()

┌────────────────┬──────────────────────┬─────────────────┬─────────────────────┐
│ missing_orders │ affected_order_value │ avg_order_value │ largest_order_value │
│     int64      │    decimal(38,2)     │     double      │    decimal(18,2)    │
├────────────────┼──────────────────────┼─────────────────┼─────────────────────┤
│             44 │             10075.78 │           229.0 │              478.94 │
└────────────────┴──────────────────────┴─────────────────┴─────────────────────┘



`discrepancy_volume` represents the total absolute mismatch across
affected orders. `affected_order_value` for missing deliveries represents
the value of orders requiring investigation and should not be interpreted
as confirmed financial loss.

## 4. Priority review examples

The largest exceptions are reviewed first to demonstrate how the
reconciliation output can support operational prioritization.

In [10]:
con.sql("""
SELECT
    order_id,
    order_amount,
    delivered_amount,
    amount_difference,
    absolute_difference
FROM mart_reconciliation_dashboard
WHERE flag_amount_discrepancy = 1
ORDER BY absolute_difference DESC
LIMIT 10
""").show()

┌──────────┬───────────────┬──────────────────┬───────────────────┬─────────────────────┐
│ order_id │ order_amount  │ delivered_amount │ amount_difference │ absolute_difference │
│ varchar  │ decimal(18,2) │  decimal(18,2)   │   decimal(18,2)   │    decimal(18,2)    │
├──────────┼───────────────┼──────────────────┼───────────────────┼─────────────────────┤
│ ORD00768 │        456.25 │           293.58 │            162.67 │              162.67 │
│ ORD00080 │        431.22 │           294.65 │            136.57 │              136.57 │
│ ORD00088 │        492.24 │           364.71 │            127.53 │              127.53 │
│ ORD00912 │        349.46 │           224.14 │            125.32 │              125.32 │
│ ORD00621 │        327.18 │           207.13 │            120.05 │              120.05 │
│ ORD00003 │        447.17 │           334.34 │            112.83 │              112.83 │
│ ORD00065 │        474.01 │           365.03 │            108.98 │              108.98 │
│ ORD00203

## 5. Dashboard mart validation

The dashboard mart must preserve one row per order while retaining
information about missing and duplicate delivery records.

In [11]:
con.sql("""
SELECT
    delivery_count,
    COUNT(*) AS orders
FROM mart_reconciliation_dashboard
GROUP BY delivery_count
ORDER BY delivery_count
""").show()

┌────────────────┬────────┐
│ delivery_count │ orders │
│     int64      │ int64  │
├────────────────┼────────┤
│              0 │     44 │
│              1 │    920 │
│              2 │     36 │
└────────────────┴────────┘



In [12]:
con.sql("""
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders,

    SUM(has_exception) AS exception_orders,
    SUM(flag_missing_delivery) AS missing_deliveries,
    SUM(flag_duplicate_delivery) AS duplicate_deliveries,
    SUM(flag_amount_discrepancy) AS amount_discrepancies,

    ROUND(SUM(discrepancy_amount), 2) AS discrepancy_volume,
    ROUND(SUM(affected_order_value), 2) AS missing_affected_value

FROM mart_reconciliation_dashboard
""").show()

┌──────────────┬───────────────┬──────────────────┬────────────────────┬──────────────────────┬──────────────────────┬────────────────────┬────────────────────────┐
│ total_orders │ unique_orders │ exception_orders │ missing_deliveries │ duplicate_deliveries │ amount_discrepancies │ discrepancy_volume │ missing_affected_value │
│    int64     │     int64     │      int128      │       int128       │        int128        │        int128        │   decimal(38,2)    │     decimal(38,2)      │
├──────────────┼───────────────┼──────────────────┼────────────────────┼──────────────────────┼──────────────────────┼────────────────────┼────────────────────────┤
│         1000 │          1000 │              137 │                 44 │                   36 │                   57 │            3106.54 │               10075.78 │
└──────────────┴───────────────┴──────────────────┴────────────────────┴──────────────────────┴──────────────────────┴────────────────────┴────────────────────────┘



The final dashboard mart preserves 1,000 unique orders and identifies
137 exception orders:

- 44 missing deliveries
- 36 duplicate deliveries
- 57 amount discrepancies

The amount-discrepancy volume is 3,106.54, while missing deliveries
affect orders with a total value of 10,075.78.

## 6. Tableau export

The validated order-level mart is exported as the data source for the
interactive Tableau reconciliation dashboard.

In [13]:
con.sql("""
COPY mart_reconciliation_dashboard
TO '../dashboard/reconciliation_dashboard.csv'
(HEADER, DELIMITER ',')
""")